# Configuration model and triangles: collect comparable measurements

We compare three graphs at the same $n$ and Poisson membership parameter
$\lambda$. The triangle graph is built by the algorithm in notebook 02.
The configuration model starts with degrees $K_i=2D_i$, where
$D_i\sim\operatorname{Poisson}(\lambda)$, and pairs its half-edges uniformly.
Its degree law matches the triangle graph **in the large sparse limit**;
finite triangle graphs can have repeated memberships or neighbours.

For every trial we record:

- **Triangle RIG:** the simple graph used throughout this project.
- **CM multigraph:** the configuration model with all loops and parallel edges.
  A loop contributes two to degree. This is the model directly covered by the
  configuration-model prediction in notebook 05.
- **CM erased:** the same pairing after removing loops and merging repeated
  edges. This checks how much the simple-graph convention affects the result.

The two CM versions are paired observations from one matching. They are not
independent samples of each other. Different trials use independent random
streams, while triangle and CM generation receive separate seeds within a trial.

Run this notebook using the project's `.venv` kernel, then open notebook 07
for plots and interpretation. Notebook 05 explains the theory before we test it.

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

%run "{NOTEBOOK_DIR}/02_algorithms.ipynb"
%run "{NOTEBOOK_DIR}/05_configuration_theory.ipynb"

PROJECT_ROOT = NOTEBOOK_DIR.parent

In [ ]:
import csv
import json
from itertools import combinations
from numbers import Integral
from time import perf_counter

import numpy as np
from joblib import Parallel, delayed

## Study settings

The default run compares the same graph sizes as the triangle study. The grid
covers its broad lambda range and puts more points near the CM prediction
$\lambda_c\approx1.28772$ and the earlier rise seen in the triangle pilot.
The points immediately around the CM threshold investigate the finite-size
transition window; they are not expected to match the limiting curve exactly.

`repeats` is the number of independent trial pairs at each $(n,\lambda)$.
`jobs` controls parallel trials. There is no nested parallel loop: one graph
size finishes and is saved before the next starts. Set `jobs=1` for a serial
check, a positive count for that many workers, or `-1` for all available CPUs.

Each size is saved as `comparison_n<size>.csv`. Existing files are protected
unless `overwrite=True`. For a new experiment, change `output_dir` and select
that directory in notebook 07. All paths start from the project root.

The raw CSV also records degree diagnostics. In particular, $2\operatorname{Poi} (\lambda)$ has mean $2\lambda$, variance $4\lambda$, and only even values;
it is not $\operatorname{Poi}(2\lambda)$. A small odd-degree fraction in the
simple projections is one way to see finite-size departures from that law.

In [ ]:
config = {
    "sizes": [1000, 3000, 10000],  # Increasing sizes, as in the triangle study.
    "lambdas": [
        0.7,
        0.9,
        1.05,
        1.10,
        1.15,
        1.20,
        1.24,
        1.26,
        1.275,
        1.285,
        1.295,
        1.31,
        1.35,
        1.4,
        1.5,
        1.8,
        2.2,
        2.5,
    ],  # A broad range plus closer points around the two observed rises.
    "repeats": 60,  # Independent trials per size and lambda; three rows per trial.
    "seed": 20261109,  # Every row saves its trial seed and graph-generation seed.
    "jobs": 4,  # Parallel by default; adjust to your machine.
    "output_dir": "results/runs/configuration_comparison",
    "overwrite": False,  # Choose another directory to retain an earlier run.
}
config

## Measure a graph and keep the degree comparison honest

The CM multigraph keeps its sampled degrees exactly. Its erased version and
our triangle graph use distinct neighbours, so their realised degrees can be
slightly smaller. We record both the nominal and realised mean degree, the
realised variance, the odd-degree fraction, and the full degree histogram.
Histograms store counts in degree order, retaining isolated vertices.

We also measure global clustering on each graph's **simple projection**:
three times the number of triangles divided by the number of connected
triples. This is the fraction of neighbour pairs that are connected. The
CM versions share this diagnostic because they share their simple projection.
It is an auxiliary simple-graph statistic, not a multigraph degree measure.

The `loops` and `parallel_excess` columns count defects before erasure. In
the triangle graph these describe the clique projection of membership slots.
For either model, removing them loses exactly twice their total in degree.

The array-based `core_mask` uses the same repeated-deletion rule as the graph
classes. Its adjacency lists can also represent loops and repeated edges,
which the simple `Graph` class intentionally cannot. Notebook tests compare
both core algorithms on simple graphs.

In [ ]:
COMPARISON_FIELDS = (
    "model",
    "n",
    "lambda",
    "replicate",
    "trial_seed",
    "seed",
    "core_size",
    "core_fraction",
    "nominal_degree_mean",
    "degree_mean",
    "degree_variance",
    "odd_degree_fraction",
    "simple_transitivity",
    "loops",
    "parallel_excess",
    "degree_histogram",
)


def simple_transitivity(adjacency) -> float:
    """Fraction of neighbour pairs that are connected, for a simple graph."""
    neighbours = [set(group) for group in adjacency]
    triples = sum(len(group) * (len(group) - 1) // 2 for group in neighbours)
    if triples == 0:
        return 0.0
    # Each triangle appears once at each of its three undirected edges.
    closed_triples = sum(
        len(neighbours[u] & neighbours[v])
        for u, group in enumerate(neighbours)
        for v in group
        if u < v
    )
    return closed_triples / triples


def comparison_measurement(
    model,
    adjacency,
    simple_adjacency,
    nominal_degree_mean,
    n,
    lam,
    replicate,
    trial_seed,
    seed,
    loops,
    parallel_excess,
) -> dict:
    """Keep the core fraction and the diagnostics needed to compare models."""
    degrees = np.array([len(group) for group in adjacency], dtype=np.int64)
    mean_degree = int(degrees.sum()) / n
    # Integer sums make this statistic independent of vertex-label ordering.
    variance = int(degrees @ degrees) / n - mean_degree**2
    core_size = int(core_mask(adjacency, 3).sum())
    return {
        "model": model,
        "n": int(n),
        "lambda": float(lam),
        "replicate": int(replicate),
        "trial_seed": int(trial_seed),
        "seed": int(seed),
        "core_size": core_size,
        "core_fraction": core_size / n,
        "nominal_degree_mean": float(nominal_degree_mean),
        "degree_mean": mean_degree,
        "degree_variance": max(0.0, variance),
        "odd_degree_fraction": float(np.count_nonzero(degrees % 2) / n),
        "simple_transitivity": simple_transitivity(simple_adjacency),
        "loops": int(loops),
        "parallel_excess": int(parallel_excess),
        "degree_histogram": json.dumps(
            np.bincount(degrees).tolist(), separators=(",", ":")
        ),
    }


def comparison_trial(n: int, lam: float, replicate: int, trial_seed: int) -> list[dict]:
    """Generate one triangle graph and one CM pairing, with separate streams."""
    if not isinstance(n, Integral) or isinstance(n, bool) or n <= 0:
        raise ValueError("n must be a positive integer")
    if not np.isfinite(lam) or lam < 0:
        raise ValueError("lambda must be finite and nonnegative")
    children = np.random.SeedSequence(trial_seed).spawn(2)
    triangle_seed, cm_seed = [
        int(child.generate_state(1, dtype=np.uint64)[0]) for child in children
    ]

    triangle = construct_special_triangle_RIG(
        n, lam, rng=np.random.default_rng(triangle_seed)
    )
    triangle_adjacency = graph_adjacency(triangle)
    triangle_loops = sum(
        u == v
        for community in triangle.communities
        for u, v in combinations(community, 2)
    )
    triangle_parallel = (
        3 * len(triangle.communities) - triangle_loops - len(triangle.edges)
    )
    triangle_nominal_mean = 2 * sum(triangle.memberships.values()) / n
    rows = [
        comparison_measurement(
            "triangle_rig",
            triangle_adjacency,
            triangle_adjacency,
            triangle_nominal_mean,
            n,
            lam,
            replicate,
            trial_seed,
            triangle_seed,
            triangle_loops,
            triangle_parallel,
        )
    ]

    rng = np.random.default_rng(cm_seed)
    degrees = 2 * rng.poisson(lam, n)
    sample = construct_configuration_model(degrees, rng=rng)
    nominal_mean = int(degrees.sum()) / n
    for model, adjacency in (
        ("cm_multigraph", sample["adjacency"]),
        ("cm_erased", sample["simple_adjacency"]),
    ):
        rows.append(
            comparison_measurement(
                model,
                adjacency,
                sample["simple_adjacency"],
                nominal_mean,
                n,
                lam,
                replicate,
                trial_seed,
                cm_seed,
                sample["loops"],
                sample["parallel_excess"],
            )
        )
    return rows


def simulate_comparison(
    n: int, lambdas, repeats: int, seed: int = 2026, jobs: int = 4
) -> list[dict]:
    """Run independent trial pairs; scheduling does not affect their seeds."""
    for name, value in (("n", n), ("repeats", repeats), ("seed", seed), ("jobs", jobs)):
        if not isinstance(value, Integral) or isinstance(value, bool):
            raise ValueError(f"{name} must be an integer")
    if n <= 0 or repeats <= 0 or seed < 0 or jobs == 0:
        raise ValueError("Use positive n/repeats, a nonnegative seed, and nonzero jobs")
    grid = [float(lam) for lam in lambdas]
    if (
        not grid
        or len(set(grid)) != len(grid)
        or any(not np.isfinite(lam) or lam < 0 for lam in grid)
    ):
        raise ValueError("Use distinct, finite, nonnegative lambda values")

    children = iter(
        np.random.SeedSequence([int(seed), int(n)]).spawn(len(grid) * repeats)
    )
    tasks = [
        (n, lam, replicate, int(next(children).generate_state(1, dtype=np.uint64)[0]))
        for lam in grid
        for replicate in range(repeats)
    ]
    batches = Parallel(n_jobs=jobs, verbose=5)(
        delayed(comparison_trial)(*task) for task in tasks
    )
    return [row for batch in batches for row in batch]


def run_comparison(config: dict, project_root: str | Path = PROJECT_ROOT) -> list[Path]:
    """Save each completed size, refusing an accidental replacement before work starts."""
    sizes = list(config["sizes"])
    if (
        not sizes
        or any(
            not isinstance(n, Integral) or isinstance(n, bool) or n <= 0 for n in sizes
        )
        or len(set(sizes)) != len(sizes)
    ):
        raise ValueError("sizes must contain distinct positive integers")
    output_dir = Path(project_root) / config["output_dir"]
    paths = [output_dir / f"comparison_n{n}.csv" for n in sizes]
    overwrite = config.get("overwrite", False)
    existing = [str(path) for path in paths if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(
            "Measurements already exist: "
            + ", ".join(existing)
            + ". Choose another output_dir or explicitly set overwrite=True."
        )
    print(
        f"Comparing {sizes} at {len(config['lambdas'])} lambda values; {config['repeats']} trials each."
    )
    for n, path in zip(sizes, paths):
        started = perf_counter()
        rows = simulate_comparison(
            n, config["lambdas"], config["repeats"], config["seed"], config["jobs"]
        )
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open(
            "w" if overwrite else "x", newline="", encoding="utf-8"
        ) as stream:
            writer = csv.DictWriter(stream, fieldnames=COMPARISON_FIELDS)
            writer.writeheader()
            writer.writerows(rows)
        print(
            f"n={n}: saved {len(rows)} observations to {path} in {perf_counter() - started:.1f} s."
        )
    return paths

## Run and save the comparison

This is the expensive cell. The default study has 3,240 independent trial
pairs, producing 9,720 measurement rows. Three rows per trial do not mean
three independent graph samples: the two CM rows deliberately share a pairing.
Each completed graph size is saved before the next starts.

Per-size random streams do not depend on size-list ordering. Changing the
lambda grid or repeat count can change individual trial seeds. The saved
`trial_seed` replays all three observations through `comparison_trial`;
`seed` identifies the generation stream for the individual graph.

In [ ]:
data_files = run_comparison(config)
data_files

## Continue with the saved-data analysis

Open `07_configuration_analysis.ipynb` and use the same data directory. It
compares the CM multigraph with its theoretical curve, checks finite-size
and erasure effects, and then compares both CM versions with the triangle
model. It reads the measurements without running these simulations again.

The main theoretical reference is [Janson and Łuczak (2007), Theorem 2.3](https://arxiv.org/pdf/math/0508453).
For the triangle degree law, see the calculation in notebook 05 and
[van der Hofstad, Komjáthy and Vadon (2021), Corollary 2.9](https://arxiv.org/pdf/1809.02514).
The latter concerns the degree law and local structure, not a theorem about
this triangle model's 3-core transition.